💡 **Environment:** `clamp-analyses`

# Supplementary Figure 3: GTEx robustness and recovery of expected tissue programs

In [ ]:
suppressPackageStartupMessages({
  library(here)
})
source(here("scripts/panels/style.R"))
source(here::here("scripts/panels/gm_ids.R"))


In [ ]:
# Original analysis and provenance guards retained; layout/export lives in supp3.R.
panel_input <- list(
  benchmark_long = here::here("output/03_model_biology/00_pseudobulk/00_benchmark/benchmark_long.csv"),
  bootstrap = here::here("output/03_model_biology/00_pseudobulk/00_benchmark/bootstrap_winrate.csv"),
  heatmap_long = here::here("output/03_model_biology/00_pseudobulk/03_b_matrix_singlecell/heatmap_long.csv"),
  corr_full = here::here("output/03_model_biology/00_pseudobulk/00_benchmark/clampfull_lv_ct_corr_full.csv"),
  assignments = here::here("output/03_model_biology/00_pseudobulk/00_benchmark/clampfull_lv_assignments.csv"),
  related_corr = here::here("output/03_model_biology/00_pseudobulk/04_hard_cell_types/related_lv_correlations.csv"),
  ari_data = here::here("output/03_model_biology/01_gtex/00_kmeans_clustering/ari_data.csv"),
  ari_comparisons = here::here("output/03_model_biology/01_gtex/00_kmeans_clustering/ari_comparisons.csv"),
  gene_fraction_ari_data = here::here("output/03_model_biology/01_gtex/00_kmeans_clustering/gene_fraction_ari_data.csv"),
  gene_fraction_ari_comparisons = here::here("output/03_model_biology/01_gtex/00_kmeans_clustering/gene_fraction_ari_comparisons.csv"),
  gtex_top1_detail = here::here("output/03_model_biology/01_gtex/07_global_alignment_rf_true_labels/top1/gtex_global_alignment_detail.csv"),
  gtex_ora_per_lv = here::here("output/03_model_biology/01_gtex/07_global_alignment_rf_true_labels/gtex_tissue_ora_per_lv.csv"),
  marker = here::here("output/03_model_biology/00_pseudobulk/02_disentangle/module_panel_ready.csv"),
  hard = here::here("output/03_model_biology/00_pseudobulk/04_hard_cell_types/hard_group_dot_ready.csv"),
  orthogonality_per_term = here::here("output/03_model_biology/01_gtex/07_geneset_orthogonality/orthogonality_per_term.csv"),
  orthogonality_summary = here::here("output/03_model_biology/01_gtex/07_geneset_orthogonality/orthogonality_summary.csv")
)
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(scales)
  library(yaml)
  library(here)
  library(grid)
  library(svglite)
  library(ragg)
})


FIG_W <- 183
FIG_H <- 190
FONT_FAMILY <- "Helvetica"
FS_TAG <- 8
FS_AXIS_TITLE <- MIN_FONT_PT
FS_AXIS_TEXT <- MIN_FONT_PT
FS_LEGEND <- MIN_FONT_PT
FS_LEGEND_TITLE <- MIN_FONT_PT
FS_DENSE_VALUE <- MIN_FONT_PT
FS_HEAT_LABEL_H <- MIN_FONT_PT
LINE_W <- 0.3

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"
GREEN_SCALE <- unlist(cfg$GREEN_SCALE)
TILE_TEXT <- c("TRUE" = "white", "FALSE" = "black")

pt_mm <- function(pt) pt / .pt

theme_nm <- function(base_size = FS_AXIS_TEXT) {
  theme_classic(base_size = base_size, base_family = FONT_FAMILY) %+replace%
    theme(
      axis.line = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks.length = unit(0.9, "pt"),
      axis.text = element_text(size = base_size, colour = "black"),
      axis.title = element_text(size = FS_AXIS_TITLE, colour = "black"),
      legend.text = element_text(size = FS_LEGEND),
      legend.title = element_text(size = FS_LEGEND_TITLE),
      legend.key = element_blank(), legend.background = element_blank(),
      panel.grid = element_blank(), plot.background = element_blank(),
      plot.margin = margin(1, 1, 1, 1, "mm")
    )
}

theme_heatmap <- function() {
  theme_nm() %+replace%
    theme(axis.line = element_blank(), axis.ticks = element_blank(),
          axis.text.x = element_text(size = FS_HEAT_LABEL_H, angle = 45,
                                     hjust = 1, vjust = 1, lineheight = 0.9),
          axis.text.y = element_text(size = FS_HEAT_LABEL_H, hjust = 1,
                                     lineheight = 0.9),
          plot.margin = margin(1, 1, 1, 1, "mm"))
}

add_tag <- function(p, label) {
  ggdraw() + draw_plot(p) +
    draw_label(label, x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
}

assign_bracket_tiers <- function(comp_df, xpos) {
  d <- copy(as.data.table(comp_df))
  d[, left := pmin(xpos[a], xpos[b])]
  d[, right := pmax(xpos[a], xpos[b])]
  d[, span := right - left]
  setorder(d, span, left)
  tiers <- list()
  d[, tier := 0L]
  for (i in seq_len(nrow(d))) {
    t <- 1L
    while (length(tiers) >= t && any(d$left[i] <= tiers[[t]]$right &
                                      d$right[i] >= tiers[[t]]$left)) t <- t + 1L
    d$tier[i] <- t
    if (length(tiers) < t) tiers[[t]] <- d[i, .(left, right)]
    else tiers[[t]] <- rbind(tiers[[t]], d[i, .(left, right)])
  }
  d[, span := NULL]
  d
}

# Bracket labels follow Figure 2a: adjusted P values as plotmath, x10^n for q < 0.001.
fmt_q <- function(q) {
  if (is.na(q)) return('"n.a."')
  if (q >= 0.001) return(sprintf('italic(q) == %.3f', q))
  parts <- strsplit(formatC(q, format = "e", digits = 1), "e")[[1]]
  sprintf("italic(q) == %s%%*%%10^displaystyle(%d)", trimws(parts[1]), as.integer(parts[2]))
}

add_brackets <- function(p, d, y_base, y_step, h, size) {
  for (i in seq_len(nrow(d))) {
    y <- y_base + (d$tier[i] - 1) * y_step
    p <- p +
      annotate("segment", x = d$left[i], xend = d$left[i], y = y, yend = y + h,
               linewidth = 0.25) +
      annotate("segment", x = d$left[i], xend = d$right[i], y = y + h, yend = y + h,
               linewidth = 0.25) +
      annotate("segment", x = d$right[i], xend = d$right[i], y = y, yend = y + h,
               linewidth = 0.25) +
      annotate("text", x = (d$left[i] + d$right[i]) / 2, y = y + h + 0.012,
               label = fmt_q(d$q[i]), size = size, vjust = 0, parse = TRUE,
               family = "sans")
  }
  p
}

shorten_gtex <- function(x) {
  x <- gsub("Adipose - Subcutaneous", "Adipose - Subcut.", x)
  x <- gsub("Adipose - Visceral \\(Omentum\\)", "Adipose - Visceral", x)
  x <- gsub("Brain - Anterior cingulate cortex \\(BA24\\)", "Brain - ACC (BA24)", x)
  x <- gsub("Brain - Caudate \\(basal ganglia\\)", "Brain - Caudate", x)
  x <- gsub("Brain - Cerebellar Hemisphere", "Brain - Cereb. hem.", x)
  x <- gsub("Brain - Frontal Cortex \\(BA9\\)", "Brain - FC (BA9)", x)
  x <- gsub("Brain - Nucleus accumbens \\(basal ganglia\\)", "Brain - NAc", x)
  x <- gsub("Brain - Putamen \\(basal ganglia\\)", "Brain - Putamen", x)
  x <- gsub("Brain - Spinal cord \\(cervical c-1\\)", "Brain - Spinal cord", x)
  x <- gsub("Brain - Substantia nigra", "Brain - Subst. nigra", x)
  x <- gsub("Cells - Cultured fibroblasts", "Cells - Fibroblasts", x)
  x <- gsub("Cells - EBV-transformed lymphocytes", "Cells - EBV lymph.", x)
  x <- gsub("Esophagus - Gastroesophageal Junction", "Esoph. - GE junction", x)
  x <- gsub("Esophagus - Mucosa", "Esoph. - Mucosa", x)
  x <- gsub("Esophagus - Muscularis", "Esoph. - Muscularis", x)
  x <- gsub("Heart - Atrial Appendage", "Heart - Atrial app.", x)
  x <- gsub("Skin - Not Sun Exposed \\(Suprapubic\\)", "Skin - Not sun exp.", x)
  x <- gsub("Skin - Sun Exposed \\(Lower leg\\)", "Skin - Sun exp.", x)
  x <- gsub("Small Intestine - Terminal Ileum", "Sm. intestine - Ileum", x)
  trimws(x)
}

# Shared Figure 2a-style boxplot: coloured boxes, white diamond means with bold mean
# labels above each box and tiered adjusted-P brackets.
benchmark_box <- function(dt, xcol, ycol, colours, comp, ylab, xlab = NULL, x_labels = waiver(),
                          y_step, angle = 35, y_top = NULL) {
  d <- copy(dt); setnames(d, c(xcol, ycol), c("x", "y"))
  means <- d[, .(mean_y = mean(y), max_y = max(y)), by = x]
  lv <- levels(d$x)
  comp <- assign_bracket_tiers(comp, setNames(seq_along(lv), lv))
  y_base <- max(means$max_y) + 0.11
  y_max <- y_base + (max(comp$tier) - 1) * y_step + 0.10
  y_need <- y_max
  if (!is.null(y_top)) y_max <- y_top
  p <- ggplot(d, aes(x, y, fill = x)) +
    geom_violin(width = 0.9, trim = TRUE, scale = "width", colour = "black", linewidth = 0.25, alpha = 0.65) +
    geom_boxplot(width = 0.18, outlier.shape = NA, colour = "black", linewidth = 0.25,
                 alpha = 0.85) +
    geom_jitter(width = 0.2, size = 0.3, shape = 21, fill = "white", colour = "#333333",
                stroke = 0.12, alpha = 0.75) +
    geom_point(data = means, aes(x = x, y = mean_y), shape = 23, size = 1.2,
               fill = "white", colour = "black", stroke = 0.35, inherit.aes = FALSE) +
    geom_text(data = means, aes(x = x, y = max_y + 0.045, label = sprintf("%.3f", mean_y)),
              size = pt_mm(5), fontface = "bold", family = "sans", inherit.aes = FALSE) +
    scale_fill_manual(values = colours, na.value = "grey70") +
    scale_x_discrete(labels = x_labels) +
    scale_y_continuous(breaks = seq(0, 1, 0.25), expand = expansion(mult = c(0.02, 0.02))) +
    coord_cartesian(ylim = c(0, y_max), clip = "on") +
    labs(x = xlab, y = ylab) +
    theme_nm() +
    theme(legend.position = "none",
          axis.text.x = element_text(angle = angle, hjust = 1))
  p <- add_brackets(p, comp, y_base, y_step, 0.015, pt_mm(5))
  attr(p, "y_need") <- y_need
  p
}

# a: RNA-seq gene-subsampling robustness.
gene_frac <- fread_gm(panel_input[["gene_fraction_ari_data"]])
gene_comp <- fread_gm(panel_input[["gene_fraction_ari_comparisons"]])
fraction_levels <- c("100%", "75%", "50%", "25%", "10%", "5%", "1%")
gene_frac[, fraction := factor(fraction, levels = fraction_levels)]
fraction_colours <- unlist(cfg$RNASEQ_FRACTION_COLORS)[fraction_levels]
# Current main report uses 25% as reference; retain all six supplied tests.
stopifnot(nrow(gene_comp) == 6L, all(gene_comp$a == "25%"))
panel_A <- benchmark_box(gene_frac, "fraction", "ari", fraction_colours, gene_comp,
                         ylab = "Adjusted Rand index (ARI)", xlab = "Genes used for RNA-seq (%)",
                         x_labels = sub("%$", "", fraction_levels), y_step = 0.10)

# b: method-level GTEx clustering benchmark.
ari <- fread_gm(panel_input[["ari_data"]])
ari_comp <- fread_gm(panel_input[["ari_comparisons"]])
ari[, method := fifelse(method == "GenomicSuperSignature", "GSSig", method)]
ari_comp[, a := fifelse(a == "GenomicSuperSignature", "GSSig", a)]
ari_comp[, b := fifelse(b == "GenomicSuperSignature", "GSSig", b)]
method_means <- ari[, .(m = mean(ari)), by = method]
pinned <- c("CLAMPfull", "PLIER", "CLAMPbase", "NMF")
pinned <- pinned[pinned %in% method_means$method]
method_levels <- c(pinned, method_means[!method %in% pinned][order(-m), method])
ari[, method := factor(method, levels = method_levels)]
method_colours <- MODEL_COLORS_RAW[method_levels]
method_colours[is.na(method_colours)] <- "grey70"
names(method_colours) <- method_levels
panel_B <- benchmark_box(ari, "method", "ari", method_colours, ari_comp,
                         ylab = "Adjusted Rand index (ARI)", y_step = 0.10)

# c: expected tissue gene sets recovered by the top SHAP module of each subtissue (Z).
# Rows are GTEx tissue gene sets and columns the top module of the matching subtissue, in
# the same order, so the outlined diagonal marks the expected gene set. Tiles are coloured
# only where the over-representation FDR is below 0.0001.
FDR_CUTOFF <- 1e-4
top1 <- fread_gm(panel_input[["gtex_top1_detail"]])
ora <- fread_gm(panel_input[["gtex_ora_per_lv"]])
top1 <- top1[order(Tissue)]
top1[, col_id := seq_len(.N)]
top1[, short := shorten_gtex(Tissue)]
ora_min <- ora[LV %in% top1$LV, .(fdr = min(p.adjust)), by = .(LV, normalized_tissue)]
grid_c <- CJ(col_id = top1$col_id, row_id = top1$col_id)
grid_c[, `:=`(LV = top1$LV[col_id], key_col = top1$normalized_true_tissue[col_id],
              key_row = top1$normalized_true_tissue[row_id])]
grid_c[, matched := key_col == key_row]
grid_c <- merge(grid_c, ora_min, by.x = c("LV", "key_row"), by.y = c("LV", "normalized_tissue"),
                all.x = TRUE)
grid_c[, z := fifelse(!is.na(fdr) & fdr < FDR_CUTOFF, pmin(-log10(fdr), 20), NA_real_)]
grid_c[, row_lab := factor(top1$short[row_id], levels = rev(top1$short))]
grid_c[, col_f := factor(col_id, levels = top1$col_id)]
stopifnot(sum(grid_c$matched) == nrow(top1))
panel_C <- ggplot(grid_c, aes(col_f, row_lab)) +
  geom_tile(fill = "white", colour = "grey92", linewidth = 0.15) +
  geom_tile(data = grid_c[!is.na(z)], aes(fill = z), colour = "white", linewidth = 0.15) +
  geom_tile(data = grid_c[matched == TRUE], fill = NA, colour = "black", linewidth = 0.35) +
  scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 20), breaks = c(0, 5, 10, 20),
                       labels = c("0", "5", "10", "≥20"), name = "-log10(FDR)") +
  scale_x_discrete(labels = setNames(paste0(top1$LV, "/", top1$short), top1$col_id),
                   expand = c(0, 0)) +
  scale_y_discrete(expand = c(0, 0)) +
  labs(x = "Top SHAP module / subtissue (columns ordered as rows)",
       y = "GTEx tissue gene set") +
  theme_heatmap() +
  theme(axis.text.x = element_text(size = FS_HEAT_LABEL_H, angle = 45, hjust = 1, vjust = 1),
        legend.position = "bottom", legend.direction = "horizontal",
        axis.title.x = element_text(margin = margin(t = 3))) +
  guides(fill = guide_colourbar(title.position = "left", barwidth = unit(30, "mm"),
                                barheight = unit(2, "mm")))

# Give a and b the same y range so the axes line up across the two panels.
y_common <- max(attr(panel_A, "y_need"), attr(panel_B, "y_need"))
panel_A <- benchmark_box(gene_frac, "fraction", "ari", fraction_colours, gene_comp,
                         ylab = "Adjusted Rand index (ARI)", xlab = "Genes used for RNA-seq (%)",
                         x_labels = sub("%$", "", fraction_levels), y_step = 0.10, y_top = y_common)
panel_B <- benchmark_box(ari, "method", "ari", method_colours, ari_comp,
                         ylab = "Adjusted Rand index (ARI)", y_step = 0.10, y_top = y_common)

In [ ]:
set.seed(123)
nm_sources(3,unlist(panel_input))
ab <- cowplot::align_plots(panel_A,panel_B,align="h",axis="tb")
page <- plot_grid(plot_grid(nm_tag(ab[[1]],"a"),nm_tag(ab[[2]],"b"),ncol=2),
 nm_tag(panel_C,"c"),ncol=1,rel_heights=c(62,150))
nm_export(list(page),3,"",height=212,heading=FALSE)